Basic Components
====

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tiktoken

### 1. Download data

In [2]:

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


url = "https://www.gutenberg.org/ebooks/2701.txt.utf-8"
repo_root = find_repo_root()
data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

text_path = data_dir / "moby_dick.txt"

if not text_path.exists():
    urlretrieve(url, text_path)

text = text_path.read_text(encoding="utf-8")

print(f"Saved to: {text_path}")
print(f"Characters: {len(text):,}")
print(text[:500])


Saved to: /home/pyml/git_vmirly/llm-recipes/data/raw/moby_dick.txt
Characters: 1,238,279
The Project Gutenberg eBook of Moby Dick; Or, The Whale
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBo


### 2. Tokenization

In [6]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text)

n_vocab = tokenizer.n_vocab

# number of unique tokens in the tokenizer's vocabulary
print(f"Number of unique tokens: {n_vocab}")

print(f"Tokens: {len(token_ids):,}")
print(f"First 10 token IDs: {token_ids[:10]}")

# decoding:
decoded_text = tokenizer.decode(token_ids[:10])
print(f"Decoded text: {decoded_text}")


Number of unique tokens: 50257
Tokens: 331,691
First 10 token IDs: [464, 4935, 20336, 46566, 286, 16540, 88, 11740, 26, 1471]
Decoded text: The Project Gutenberg eBook of Moby Dick; Or


### 3. Embeddings

In [11]:
n_embed = 768
context_length = 1024

In [10]:
import torch
import torch.nn as nn

token_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=n_embed
)

print(f"Token embeddings shape: {token_embeddings.weight.shape}")

# testing:
token_embeddings(torch.tensor(token_ids[:10]))

Token embeddings shape: torch.Size([50257, 768])


tensor([[ 0.5068, -1.2898,  0.8113,  ..., -0.9357, -1.4411, -0.8971],
        [-1.0285, -1.4554, -1.0485,  ...,  1.3176,  1.3046,  2.0392],
        [-0.0542, -1.7150, -1.2420,  ...,  1.2358,  0.1843, -0.5916],
        ...,
        [-0.8588,  0.0461,  0.8433,  ...,  1.9706,  1.3442, -1.2125],
        [-0.3512,  0.3448, -0.6046,  ..., -1.4364, -0.2131, -0.4496],
        [-0.8223,  1.1525, -0.8771,  ...,  0.3721, -0.4495, -0.5532]],
       grad_fn=<EmbeddingBackward0>)

In [13]:
pos_embeddings = nn.Embedding(
    num_embeddings=context_length, embedding_dim=n_embed
)

pos_embeddings(torch.arange(10)).shape

torch.Size([10, 768])

## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py